# Lesson 12: Save, reload, and serve predictions

Package model weights, tokenizer metadata, and configuration into a checkpoint, then build and test a bounded prediction handler locally.

**How to run:** Select a Python kernel with PyTorch installed, then run each code cell from top to bottom with **Shift+Enter**. This notebook is self-contained; no other notebook needs to run first. Restart the kernel and run from the top to reset the experiment.

**Source:** This lesson was developed from the [reference conversation's roadmap](https://chatgpt.com/share/6aa56bca-4a1c-83e9-9153-1edcc7ff7e40). The reference supplies Lesson 1 and a topic outline; Lessons 2–12 are newly written implementations of those topics. Small examples demonstrate the mechanics; they are not trained assistants.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
# Small tensors can be slower with many CPU threads.
torch.set_num_threads(1)
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)


## Load and tokenize text

We reuse `input.txt`, resolving it from the notebook folder or repository root. `B` means batch size, `T` means context length, and `C` means vector width. Our file is only 80 characters, so the validation scores are noisy and text generation will be limited.


In [ ]:
input_path = Path('input.txt')
if not input_path.is_file():
    input_path = Path('chatgpt/input.txt')
text = input_path.read_text(encoding='utf-8')
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return ''.join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.8 * len(data))
train_data, val_data = data[:split], data[split:]
block_size = min(8, len(train_data) - 1, len(val_data) - 1)
if block_size < 1:
    raise ValueError('input.txt needs enough text for train and validation sequences.')
batch_size = 4
print('Characters:', vocab_size, '| train:', len(train_data), '| validation:', len(val_data))
print('Context length:', block_size)


## Draw random batches

Choose starting positions within one split. Targets are the same window shifted right by one token. No window crosses from training into validation. The tokenizer vocabulary uses the full text so every validation character has an ID; model weights are updated only on training tokens.


In [ ]:
def get_batch(split_name='train'):
    if split_name not in ('train', 'val'):
        raise ValueError("Choose 'train' or 'val'.")
    source = train_data if split_name == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch()
print('Input shape:', xb.shape, '| target shape:', yb.shape)
print('Input :', repr(decode(xb[0])))
print('Target:', repr(decode(yb[0])))
assert torch.equal(xb[:, 1:], yb[:, :-1])


## Causal multi-head attention

Each head compares queries to keys, then combines value vectors. A triangular mask prevents reading future tokens. This is the implementation developed in Lessons 3–4, included here so this notebook runs independently.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.head_size = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.projection = nn.Linear(width, width)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('causal_mask', torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # Give each head its own vector slice: [B, heads, T, head_size].
        q = q.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        k = k.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        v = v.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        weights = self.dropout(F.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().reshape(B, T, C)
        return self.projection(out)


## Transformer block

LayerNorm normalizes each token's features. Residual additions let information pass around attention and the MLP. The MLP expands each token vector, applies a nonlinear function, and projects it back.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(width)
        self.attention = CausalSelfAttention(width, heads, context_length, dropout)
        self.ln2 = nn.LayerNorm(width)
        self.mlp = nn.Sequential(
            nn.Linear(width, 4 * width), nn.GELU(),
            nn.Linear(4 * width, width), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


## Token and position embeddings → vocabulary scores

Token embeddings describe characters; learned position embeddings distinguish their positions. The final linear layer predicts the next character at every position. Cross entropy consumes raw scores (logits).


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, context_length, width=32, heads=4, layers=2):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context_length, width)
        self.blocks = nn.Sequential(*[
            TransformerBlock(width, heads, context_length) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.context_length:
            raise ValueError('Sequence exceeds context length.')
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        logits = self.lm_head(self.final_norm(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

model = TinyGPT(vocab_size, block_size).to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))


## Train a small model

This cell continues training if rerun. Rerun the model-creation cell first for a fresh model. The small default run teaches the workflow; useful generation needs substantially more text and training.


In [ ]:
training_steps = 120
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()
for step in range(training_steps):
    x, y = get_batch()
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 40 == 0 or step == training_steps - 1:
        print(f'step {step}: loss {loss.item():.4f}')


## Save the complete model contract

Weights alone are insufficient: loading also needs the architecture configuration and the exact ID-to-character mapping. This cell writes `artifacts/lesson12_tiny_gpt.pt` next to the training text. Rerunning replaces this lesson's checkpoint.


In [ ]:
artifact_dir = input_path.resolve().parent / 'artifacts'
artifact_dir.mkdir(exist_ok=True)
checkpoint_path = artifact_dir / 'lesson12_tiny_gpt.pt'
config = dict(vocab_size=vocab_size, context_length=block_size, width=32, heads=4, layers=2)
checkpoint = {
    'format_version': 1,
    'config': config,
    'chars': chars,
    'model_state': {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()},
}
torch.save(checkpoint, checkpoint_path)
print('Saved:', checkpoint_path)


## Reload and verify identical predictions

Use `weights_only=True` for this tensor-and-primitive checkpoint. A checkpoint still needs compatible model class definitions, supplied above.


In [ ]:
loaded = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
assert loaded['format_version'] == 1
reloaded = TinyGPT(**loaded['config'])
reloaded.load_state_dict(loaded['model_state'])
reloaded.eval()
model.eval()
probe, _ = get_batch()
with torch.no_grad():
    expected = model(probe)[0].cpu()
    actual = reloaded(probe.cpu())[0]
assert torch.allclose(expected, actual, atol=1e-6)
print('Reloaded predictions match.')


## Define a prediction handler

A web service can call this function after decoding a JSON request. The handler loads no files per request, limits generation length, validates the prompt, and uses the checkpoint's tokenizer. It uses greedy decoding for reproducible checks. This cell does not start a server or deploy cloud resources.


In [ ]:
loaded_chars = loaded['chars']
loaded_stoi = {ch: i for i, ch in enumerate(loaded_chars)}

@torch.no_grad()
def predict(request):
    if not isinstance(request, dict):
        raise ValueError('Request must be a dictionary.')
    prompt = request.get('prompt')
    count = request.get('max_new_tokens', 40)
    if not isinstance(prompt, str) or not 1 <= len(prompt) <= 1000:
        raise ValueError('prompt must contain 1–1000 characters.')
    if type(count) is not int or not 0 <= count <= 200:
        raise ValueError('max_new_tokens must be an integer from 0 to 200.')
    unknown = sorted(set(prompt) - set(loaded_stoi))
    if unknown:
        raise ValueError(f'Characters outside tokenizer vocabulary: {unknown!r}')
    ids = torch.tensor([[loaded_stoi[ch] for ch in prompt]], dtype=torch.long)
    reloaded.eval()
    for _ in range(count):
        logits, _ = reloaded(ids[:, -reloaded.context_length:])
        ids = torch.cat([ids, logits[:, -1].argmax(dim=-1, keepdim=True)], dim=1)
    return {'text': ''.join(loaded_chars[i] for i in ids[0].tolist()), 'new_tokens': count}

response = predict({'prompt': text[:1], 'max_new_tokens': 30})
print(response)
assert len(response['text']) == 31
for bad_request in ({'prompt': ''}, {'prompt': text[:1], 'max_new_tokens': 201}):
    try:
        predict(bad_request)
    except ValueError as error:
        print('Rejected invalid request:', error)
    else:
        raise AssertionError('Invalid request was accepted.')


## From a function to a deployed service

The next engineering steps are to wrap `predict` in an HTTP endpoint, package the model code and pinned dependencies, and load the checkpoint once at process startup. For an AWS exercise, a container can load its versioned checkpoint from object storage and expose a health check and prediction endpoint. Choose the hosting service after measuring memory, latency, and concurrency needs.

Track model version, request latency, failures, and token counts. Keep evaluation examples to check each model release. A notebook demonstration does not establish deployment readiness or useful answer quality; those require testing with representative data and traffic.


## Try it yourself

Restart the kernel and load the saved checkpoint after running only the imports and class-definition cells (skip model training). Add a model-version field to the response. Measure latency for 10 versus 100 generated tokens before selecting hosting hardware.
